# Frontier Local Metacognition Submission Notebook

This notebook packages the frozen no-API local/open-weight benchmark result.

**Core claim:** small open models can avoid bluffing while still failing metacognitively by collapsing ordinary hidden-state uncertainty into `ESCALATE` instead of distinguishing when `ABSTAIN` is correct.

In [ ]:
from pathlib import Path
import json
from IPython.display import Markdown, SVG, display

ROOT = Path.cwd()
TASK_FILE = ROOT / 'frontier_tasks_metacog.jsonl'
QWEN_RESULTS = ROOT / 'qwen_results.json'
SMOLLM_RESULTS = ROOT / 'smollm_results.json'
FIGURE = ROOT / 'frontier_local_metacognition_behavior.svg'

tasks = [json.loads(line) for line in TASK_FILE.read_text(encoding='utf-8').splitlines() if line.strip()]
qwen = json.loads(QWEN_RESULTS.read_text(encoding='utf-8'))
smollm = json.loads(SMOLLM_RESULTS.read_text(encoding='utf-8'))
print(f'Loaded {len(tasks)} tasks and 2 model result files.')

## Action Semantics

- `COMMIT`: evidence is sufficient for a conclusion.
- `ABSTAIN`: evidence is insufficient, but there is no contradiction or trust collapse.
- `ESCALATE`: contradiction, trust failure, or model insufficiency requires outside review.

In [ ]:
task_order = [
    'hidden_state_uncertainty',
    'adversarial_trust',
    'overflow_mismatch',
    'clear_commit',
]
examples = {}
for task in tasks:
    examples.setdefault(task['task_type'], task)

display(Markdown('## Example Tasks'))
for task_type in task_order:
    task = examples[task_type]
    signals = '\n'.join(f"- {signal}" for signal in task['signals'][:4])
    display(Markdown(
        f"### {task_type}\n"
        f"**Task ID:** `{task['task_id']}`  \\n"
        f"**Source:** `{task['source_domain']}` / `{task['source_task_family']}` / `{task['source_split']}`  \\n"
        f"**Gold action:** `{task['gold_action']}`  \\n"
        f"**Context:** {task['context']}\n\n"
        f"**Signals**\n{signals}"
    ))

In [ ]:
def overall_row(label, payload):
    s = payload['summary']['summary']
    return (
        f"| {label} | {s['final_accuracy']:.3f} | {s['commit_accuracy']:.2f} | "
        f"{s['abstain_rate']:.2f} | {s['bluff_rate']:.2f} | {s['escalation_rate']:.2f} | {s['silent_failure_rate']:.2f} |"
    )

table = [
    '## Compact Results Table',
    '',
    '| Model | Final Acc | Commit Acc | Abstain | Bluff | Escalate | Silent Failure |',
    '|---|---:|---:|---:|---:|---:|---:|',
    overall_row('Qwen', qwen),
    overall_row('SmolLM', smollm),
]
display(Markdown('\n'.join(table)))

In [ ]:
display(Markdown('## Behavior Figure'))
display(SVG(filename=str(FIGURE)))

## Conclusion

- Both local/open-weight models achieved `0` bluffing and `0` silent failure.
- Both handled `adversarial_trust` and `overflow_mismatch` cleanly by escalating.
- The main failure mode is ordinary hidden-state uncertainty, where both models over-escalate instead of separating `ABSTAIN` from `ESCALATE`.
- `SmolLM` is stronger on clear commits, while `Qwen` is slightly closer to the intended metacognitive distinction because it produces at least some true `ABSTAIN`.